In [ ]:
import os
from contextlib import nullcontext

import numpy as np
import torch
from PIL import Image, ImageDraw
from tqdm import tqdm
from diffusers import StableDiffusionXLControlNetPipeline, UniPCMultistepScheduler

from cnet_dataset_sdxl import HSIControlNetDataset
from warp_and_controlnet_model import WarpAndControlNetModel

def tensor_to_pil(tensor_chw):
    arr = tensor_chw.detach().cpu().permute(1, 2, 0).numpy()
    arr = np.clip((arr + 1.0) * 127.5, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

def vis48(ch48):
    ch48 = ch48.astype(np.float32)
    ch48 = (ch48 - ch48.min()) / (ch48.max() - ch48.min() + 1e-6)
    rgb = ch48[:3].transpose(1, 2, 0)
    rgb = (rgb * 255).astype(np.uint8)
    return Image.fromarray(rgb)

def add_title(img: Image.Image, title: str, title_h: int = 32):
    w, h = img.size
    canvas = Image.new("RGB", (w, h + title_h), color=(0, 0, 0))
    canvas.paste(img, (0, title_h))
    draw = ImageDraw.Draw(canvas)
    draw.text((10, 8), title, fill=(255, 255, 255))
    return canvas

def split_2x2(img: Image.Image):
    w, h = img.size
    assert w % 2 == 0 and h % 2 == 0, f"期望偶数尺寸，得到: {(w, h)}"
    hw, hh = w // 2, h // 2
    tl = img.crop((0, 0, hw, hh))
    tr = img.crop((hw, 0, w, hh))
    bl = img.crop((0, hh, hw, h))
    br = img.crop((hw, hh, w, h))
    return {"tl": tl, "tr": tr, "bl": bl, "br": br}

/home/viplab/anaconda3/envs/cnet2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
OUTPUT_DIR = "./output_sdxl"
compare_dir = os.path.join(OUTPUT_DIR, "compare")
result_dir = os.path.join(OUTPUT_DIR, "result")
os.makedirs(compare_dir, exist_ok=True)
os.makedirs(result_dir, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
weight_dtype = torch.float16 if device == "cuda" else torch.float32

controlnet = WarpAndControlNetModel.from_pretrained(
    CKPT_DIR,
    torch_dtype=weight_dtype,
    low_cpu_mem_usage=False,
    use_safetensors=True,
    device_map=None,
)

pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    BASE_MODEL,
    controlnet=controlnet,
    torch_dtype=weight_dtype,
    use_safetensors=True,
    safety_checker=None,
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

if device == "cuda":
    pipe.enable_model_cpu_offload()
else:
    pipe = pipe.to(device)

dataset = HSIControlNetDataset(root_dir=ROOT_DIR, image_size=IMAGE_SIZE)
n = min(NUM_SAMPLES, len(dataset))
generator = torch.Generator(device=device).manual_seed(SEED)

for idx in tqdm(range(n), desc="Infer"):
    sample = dataset[idx]

    gt_img = tensor_to_pil(sample["pixel_values"])
    cond_tensor = sample["conditioning_pixel_values"]
    cond_img = vis48(cond_tensor.detach().cpu().numpy())

    cond_batch = cond_tensor.unsqueeze(0).to(device=device, dtype=weight_dtype)

    autocast_ctx = (
        torch.autocast(device_type="cuda", dtype=torch.float16) if device == "cuda" else nullcontext()
    )
    with torch.inference_mode(), autocast_ctx:
        pred = pipe(
            prompt=sample["text"],
            image=cond_batch,
            num_inference_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            generator=generator,
        ).images[0]

    # compare: 不裁切，且加标注
    gt_t = add_title(gt_img, "GT")
    cond_t = add_title(cond_img, "COND")
    pred_t = add_title(pred, "INFERENCE")

    w, h = gt_t.size
    compare_canvas = Image.new("RGB", (w * 3, h), color=(255, 255, 255))
    compare_canvas.paste(gt_t, (0, 0))
    compare_canvas.paste(cond_t, (w, 0))
    compare_canvas.paste(pred_t, (w * 2, 0))
    compare_canvas.save(os.path.join(compare_dir, f"{idx:04d}_compare.png"))

    # result: 对推理图做 2x2 裁切后保存
    crops = split_2x2(pred)
    crops["tl"].save(os.path.join(result_dir, f"{idx:04d}_tl.png"))
    crops["tr"].save(os.path.join(result_dir, f"{idx:04d}_tr.png"))
    crops["bl"].save(os.path.join(result_dir, f"{idx:04d}_bl.png"))
    crops["br"].save(os.path.join(result_dir, f"{idx:04d}_br.png"))

print(f"✅ compare 已保存到: {compare_dir}")
print(f"✅ result(裁切后) 已保存到: {result_dir}")

Warper weights loaded from /data/mv_model_0_with_drop/checkpoint-6500/controlnet/warper.pt


Warper weights loaded from /data/mv_model_0_with_drop/checkpoint-6500/controlnet/warper.pt


Keyword arguments {'safety_checker': None} are not expected by StableDiffusionXLControlNetPipeline and will be ignored.
Loading pipeline components...: 100%|██████████| 7/7 [00:02<00:00,  2.40it/s]


Warper weights loaded from /data/mv_model_0_with_drop/checkpoint-6500/controlnet/warper.pt


Keyword arguments {'safety_checker': None} are not expected by StableDiffusionXLControlNetPipeline and will be ignored.
Loading pipeline components...: 100%|██████████| 7/7 [00:02<00:00,  2.40it/s]


...

Warper weights loaded from /data/mv_model_0_with_drop/checkpoint-6500/controlnet/warper.pt


Keyword arguments {'safety_checker': None} are not expected by StableDiffusionXLControlNetPipeline and will be ignored.
Loading pipeline components...: 100%|██████████| 7/7 [00:02<00:00,  2.40it/s]


dataset name: /data/test/
[Dataset] 加载缓存...
folders len 16000
groups  len 4000


100%|██████████| 20/20 [00:06<00:00,  3.00it/s]
/home/viplab/anaconda3/envs/cnet2/lib/python3.10/site-packages/diffusers/pipelines/controlnet/pipeline_controlnet_sd_xl.py:928: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(
/home/viplab/anaconda3/envs/cnet2/lib/python3.10/site-packages/diffusers/image_processor.py:148: RuntimeWarning: invalid value encountered in cast
  images = (images * 255).round().astype("uint8")
Infer: 100%|██████████| 10/10 [01:25<00:00,  8.56s/it]

✅ compare 已保存到: ./test_result/compare
✅ result(裁切后) 已保存到: ./test_result/result
